<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/01_llm_api_basics/llm_api_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · LLM API Basics: First Steps Toward Agentic AI

Calling an LLM from code, streaming responses, and **chaining calls** so the output of one becomes the input of the next: the core building block of every agent.

Uses **Groq** (free tier, very fast inference). The notebook picks the best available models automatically, since Groq retires model names over time.

> 🔑 Add `GROQ_API_KEY` to Colab Secrets (🔑 icon in the left sidebar). Get a free key at [console.groq.com](https://console.groq.com/keys).

In [6]:
!pip install -q groq

In [7]:
import os
from google.colab import userdata
from groq import Groq
from IPython.display import Markdown, display

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("✅ Groq client ready")

✅ Groq client ready


In [8]:
# ─── Pick a model that your Groq account can access ──────────────────────────
# Groq retires models over time (e.g. llama-3.1-8b-instant was deprecated in 2026),
# so we list what's available and choose the first match from a preference list.
available = sorted(m.id for m in client.models.list().data)
print("Available models:", available)

def pick(preferences):
    for name in preferences:
        if name in available:
            return name
    chat_models = [m for m in available if not any(x in m for x in ("whisper", "tts", "guard", "embed"))]
    return chat_models[0]

FAST_MODEL = pick(["openai/gpt-oss-20b", "llama-3.1-8b-instant", "llama3-8b-8192"])
SMART_MODEL = pick(["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b"])
print(f"\n✅ Fast model : {FAST_MODEL}\n✅ Smart model: {SMART_MODEL}")

Available models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']

✅ Fast model : openai/gpt-oss-20b
✅ Smart model: openai/gpt-oss-120b


## 1. A first call
Messages use the familiar `role` / `content` format shared by OpenAI-compatible APIs.

In [9]:
messages = [{"role": "user", "content": "What is 2 + 2? Answer in one word."}]
response = client.chat.completions.create(model=FAST_MODEL, messages=messages)
print(response.choices[0].message.content)

four


## 2. Streaming a long response
`stream=True` returns tokens as they're generated, which is how chat UIs show text "typing" in real time.

> Note: parameters like `reasoning_effort` only work with reasoning models. Passing it to a non-reasoning model returns a `400 BadRequestError`, so it's omitted here.

In [10]:
stream = client.chat.completions.create(
    model=FAST_MODEL,
    messages=[{"role": "user", "content": "Write a short article (about 250 words) on use cases of AI in healthcare."}],
    temperature=0.7,
    max_completion_tokens=600,
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

**Artificial Intelligence in Healthcare: From Diagnosis to Personalized Care**

Artificial intelligence (AI) is reshaping modern medicine, turning data into actionable insights and streamlining workflows across the care continuum. In diagnostics, deep‑learning algorithms now analyze radiographs, CT scans, and MRIs with accuracy comparable to expert radiologists, flagging subtle lesions that might otherwise be missed. AI‑powered pathology tools can quantify tumor margins and predict metastatic potential from histology slides in a fraction of the time required for manual review.

Personalized medicine is another frontier. Machine‑learning models sift through genomic, proteomic, and electronic health record (EHR) data to identify optimal drug regimens for individual patients, reducing trial‑and‑error and improving outcomes. In drug discovery, AI accelerates the identification of promising molecular candidates by simulating interactions and predicting toxicity before costly laboratory test

## 3. Chaining two calls
The model writes a hard question, then a second call answers it. Two LLM calls, wired together in code.

In [11]:
MODEL = SMART_MODEL

def ask(prompt, model=MODEL):
    r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

question = ask("Propose a hard, challenging question to assess someone's IQ. Respond only with the question.")
print("❓", question)
display(Markdown(ask(question)))

❓ A sequence of numbers follows the pattern: 2, 12, 1112, 3112, 132112, ... What are the next two numbers in the sequence?


The rule is the classic “look‑and‑say” description:

* 2 → “one 2” → 12  
* 12 → “one 1, one 2” → 1112  
* 1112 → “three 1s, one 2” → 3112  
* 3112 → “one 3, two 1s, one 2” → 132112  

So each term describes the previous term by stating how many of each digit appear consecutively.

Applying that to the last given term:

**132112** → “one 1, one 3, one 2, two 1s, one 2” → **1113122112**

Now describe this new term:

**1113122112** → “three 1s, one 3, one 1, two 2s, two 1s, one 2” → **311311222112**

Hence the next two numbers in the sequence are  

**1113122112** and **311311222112**.

## 4. Exercise: an agentic workflow in three calls

1. Ask the LLM for a **business area** ripe for an agentic AI solution
2. Ask for the biggest **pain point** in that area
3. Ask it to propose an **agentic AI solution** for that pain point

Each step's output feeds the next prompt. This is the simplest form of a multi-step agent pipeline.

In [12]:
business_area = ask("Name ONE business area that is ripe for an agentic AI solution. "
                    "Reply with the area name and one sentence explaining why.")
display(Markdown(f"### 🏢 Business area\n{business_area}"))

pain_point = ask(f"In this business area: {business_area}\n\n"
                 "What is the single most painful, expensive problem professionals face? "
                 "Describe it in 3-4 sentences.")
display(Markdown(f"### 😣 Pain point\n{pain_point}"))

solution = ask(f"Pain point: {pain_point}\n\n"
               "Propose an agentic AI solution: which agents, which tools each agent uses, "
               "and how they hand work to each other. Keep it under 200 words.")
display(Markdown(f"### 🤖 Agentic solution\n{solution}"))

### 🏢 Business area
**Supply Chain Management** – because coordinating complex logistics, demand forecasting, and dynamic routing can be automated and continuously optimized by an autonomous AI that reacts in real time to market shifts, disruptions, and inventory data.

### 😣 Pain point
The single most painful and costly problem in supply‑chain management is **demand‑supply mismatch caused by inaccurate forecasting and slow reaction to disruptions**. When forecasts deviate from reality, companies either over‑stock (tying up capital, incurring warehousing and obsolescence costs) or under‑stock (triggering stock‑outs, lost sales, and emergency expediting fees). Because the data sources—market trends, weather events, supplier lead times, and real‑time inventory—are fragmented and change rapidly, traditional planning cycles cannot adjust quickly enough, turning a forecasting error into a cascade of amplified costs across production, transportation, and customer service. The result is a perpetual “bullwhip” effect that erodes margins and makes the entire supply network vulnerable to even modest disturbances.

### 🤖 Agentic solution
**Agentic AI Supply‑Chain Orchestrator (≈170 words)**  

| Agent | Core Role | Tools & Data |
|-------|-----------|--------------|
| **Forecasting Agent** | Generates ultra‑short‑term demand signals and updates baseline forecasts. | LLM‑augmented time‑series (Prophet/DeepAR), market‑trend APIs, social‑media sentiment, SKU‑level sales history. |
| **Disruption‑Detection Agent** | Monitors external shocks and flags variance from forecast. | Real‑time weather feeds, logistics‑carrier status, news‑scraping LLM, anomaly‑detection on sensor/IoT streams. |
| **Inventory‑Optimization Agent** | Re‑optimises safety stock & replenishment plans instantly. | Mixed‑integer programming solver (Gurobi/CPLEX), Monte‑Carlo scenario engine, ERP/WMS API. |
| **Execution Agent** | Issues purchase orders, production schedules, and transport bookings. | RPA bots, EDI connectors, chat‑ops for human approval, auto‑routing APIs. |
| **Collaboration Hub** (Mediator) | Routes insights, resolves conflicts, and surfaces “human‑in‑the‑loop” decisions. | Knowledge‑graph knowledge base, LLM‑driven reasoning, UI dashboard, Slack/Teams bots. |

**Workflow**  
1. Forecasting Agent pushes a 24‑h demand forecast to the Hub.  
2. Disruption‑Detection Agent continuously scans feeds; on any deviation > 5 % it pushes an alert to the Hub.  
3. Hub reconciles forecast vs. alert, selects the most credible signal, and triggers the Inventory‑Optimization Agent.  
4. Optimizer returns revised order‑qty/production‑lot sizes to the Hub, which assigns Execution Agent to create/modify PO/DRP records via ERP.  
5. All steps are logged in the Hub; when confidence falls below a threshold, the Hub prompts a human manager through the dashboard.  

Result: near‑real‑time, closed‑loop demand‑supply alignment that eliminates the bullwhip cascade.